# EDA Summary
- [EDA Summary](#eda-summary)
- [Base Analysis](#base-analysis)
- [Dates](#check-dates)
- [Location](#location)
- [Flat Characteristics](#flat-characteristics)
- [Lease Characteristics](#lease-characteristics)
- [Clean Function](#clean-function)
- [Join Composite Keys](#join-composite-keys)

Written By: Keltonn [A0222591J]

AI Used: Claude Sonnet 5.5 for API Lookup

In this notebook, we look at the baseline dataframe for the train and test period and do some analysis on the dataset, such as checking null counts etc.

In [1]:
import pandas as pd
import numpy as np
import scipy as sp
import os, sys
from pathlib import Path
_here = Path(os.environ.get('RENT_PROJECT_ROOT') or Path.cwd()).resolve()
sys.path.insert(0, str(next(p for p in [_here, *_here.parents] if (p / 'data' / 'config.py').exists())))      # final_report: data/config.py holds every path
from data import config as cfg
ROOT = cfg.ROOT

# Base Analysis

- [EDA Summary](#eda-summary)
- [Base Analysis](#base-analysis)
- [Dates](#check-dates)
- [Location](#location)
- [Flat Characteristics](#flat-characteristics)
- [Lease Characteristics](#lease-characteristics)
- [Clean Function](#clean-function)
- [Join Composite Keys](#join-composite-keys)

Our basic analysis of the dataset reveals two things:
1. There are no null values for the basic rental data for both the train and test set, which is good.
2. There are some duplicate rows where all the values for all the columns are the same.

Regarding the duplicates, we originally wanted to drop them, but there were some valid reasons that explain why these flat units were duplicating. One of the reasons given in the Canvas discussion is that the rent is denoted in the block level, so some of the units may have the same characteristics and are selling at the same price within the block.

As we cannot verify why the data is duplicating, we decide to leave the duplicates in the train and test set. 

In [2]:
train = pd.read_csv(f'{cfg.ORIGINAL_DIR}/train.csv')
test = pd.read_csv(f'{cfg.ORIGINAL_DIR}/test.csv')

In [3]:
# Check null rates
nulldf = pd.DataFrame({
    'null_count': train.isnull().sum(),
    'null_rate_%': (train.isnull().mean() * 100).round(2)
}).sort_values('null_rate_%', ascending=False)
print('====== Train Null Rates ======')
print(nulldf)

nulldf = pd.DataFrame({
    'null_count': test.isnull().sum(),
    'null_rate_%': (test.isnull().mean() * 100).round(2)
}).sort_values('null_rate_%', ascending=False)
print('====== Test Null Rates ======')
print(nulldf)

# Duplicates
print('====== Unique rows ======')
print(f"Train unique row ratio: {train.drop_duplicates().shape[0]} / {train.shape[0]}")
print(f"Test unique row ratio: {test.drop_duplicates().shape[0]} / {test.shape[0]}")

====== Train Null Rates ======
                     null_count  null_rate_%
RENT_APPROVAL_DATE            0          0.0
TOWN                          0          0.0
BLOCK                         0          0.0
STREET                        0          0.0
FLAT_TYPE                     0          0.0
FLAT_MODEL                    0          0.0
FLOOR_AREA_SQM                0          0.0
FURNISHED                     0          0.0
LEASE_COMMENCE_DATE           0          0.0
FEE                           0          0.0
MONTHLY_RENT                  0          0.0
====== Test Null Rates ======
                     null_count  null_rate_%
RENT_APPROVAL_DATE            0          0.0
TOWN                          0          0.0
BLOCK                         0          0.0
STREET                        0          0.0
FLAT_TYPE                     0          0.0
FLAT_MODEL                    0          0.0
FLOOR_AREA_SQM                0          0.0
FURNISHED                     0        

In [4]:
# Count duplicate rows (extra copies only)
train.duplicated().sum()

# Group identical rows together so the copies sit next to each other
train[train.duplicated(keep=False)].sort_values(list(train.columns))

,RENT_APPROVAL_DATE,TOWN,BLOCK,STREET,FLAT_TYPE,FLAT_MODEL,FLOOR_AREA_SQM,FURNISHED,LEASE_COMMENCE_DATE,FEE,MONTHLY_RENT
66012,2022-11,bukit merah,80c,telok blangah street 31,3-room,premium apartment,68.0,yes,2013,0.0,3000
66598,2022-11,bukit merah,80c,telok blangah street 31,3-room,premium apartment,68.0,yes,2013,0.0,3000
86589,2023-06,bukit merah,16,cantonment close,3-room,model a,60.0,yes,2003,0.0,3400
87763,2023-06,bukit merah,16,cantonment close,3-room,model a,60.0,yes,2003,0.0,3400
96976,2023-10,toa payoh,108,potong pasir avenue 1,3-room,model a,74.0,yes,1984,0.0,3500
...,...,...,...,...,...,...,...,...,...,...,...
149582,2025-03,toa payoh,116,lorong 2 toa payoh,3-room,improved,63.0,yes,1970,0.0,3000
147445,2025-03,woodlands,511,woodlands drive 14,5-room,premium apartment,125.0,yes,1999,0.0,3100
148118,2025-03,woodlands,511,woodlands drive 14,5-room,premium apartment,125.0,yes,1999,0.0,3100
147881,2025-03,yishun,111,yishun ring road,4-room,new generation,94.0,yes,1982,0.0,3000


# Check Dates
- [EDA Summary](#eda-summary)
- [Base Analysis](#base-analysis)
- [Dates](#check-dates)
- [Location](#location)
- [Flat Characteristics](#flat-characteristics)
- [Lease Characteristics](#lease-characteristics)
- [Clean Function](#clean-function)
- [Join Composite Keys](#join-composite-keys)

Here we check the date and time of the rent approval dates. There does not seem to be anything out of the ordinary here, and the time data seems to be clean.

The train period is from 2021-01 to 2025-03. The test period is from 2025-03 to 2026-07. The train test split is on the temporal level, so this is a forecasting problem built using a regression model.

In [5]:
train['RENT_APPROVAL_YEAR'] = train['RENT_APPROVAL_DATE'].apply(lambda x: x.split('-')[0])
train['RENT_APPROVAL_MONTH'] = train['RENT_APPROVAL_DATE'].apply(lambda x: x.split('-')[1])
train

,RENT_APPROVAL_DATE,TOWN,BLOCK,STREET,FLAT_TYPE,FLAT_MODEL,FLOOR_AREA_SQM,FURNISHED,LEASE_COMMENCE_DATE,FEE,MONTHLY_RENT,RENT_APPROVAL_YEAR,RENT_APPROVAL_MONTH
0,2021-01,punggol,604a,punggol road,4 room,model a,93.0,yes,2012,0.0,1800,2021,01
1,2021-01,sengkang,290c,compassvale crescent,5 room,improved,110.0,yes,2001,0.0,2000,2021,01
2,2021-01,queenstown,5,ghim moh road,3 room,improved,65.0,yes,1976,0.0,2000,2021,01
3,2021-01,sengkang,436b,fernvale road,4 room,model a,96.0,yes,2010,0.0,1400,2021,01
4,2021-01,sembawang,467b,admiralty drive,4 room,premium apartment,102.0,yes,2001,0.0,1800,2021,01
...,...,...,...,...,...,...,...,...,...,...,...,...,...
149995,2025-03,bukit batok,220,bukit batok east avenue 3,3-room,new generation,67.0,yes,1985,0.0,2500,2025,03
149996,2025-03,queenstown,79,dawson road,5-room,improved,107.0,yes,2020,0.0,5300,2025,03
149997,2025-03,toa payoh,17,lorong 7 toa payoh,3-room,improved,67.0,yes,1971,0.0,2650,2025,03
149998,2025-03,sembawang,360a,Admiralty Drive,4-room,premium apartment,94.0,yes,2001,0.0,3300,2025,03


In [6]:
test['RENT_APPROVAL_YEAR'] = test['RENT_APPROVAL_DATE'].apply(lambda x: x.split('-')[0])
test['RENT_APPROVAL_MONTH'] = test['RENT_APPROVAL_DATE'].apply(lambda x: x.split('-')[1])
test

,RENT_APPROVAL_DATE,TOWN,BLOCK,STREET,FLAT_TYPE,FLAT_MODEL,FLOOR_AREA_SQM,FURNISHED,LEASE_COMMENCE_DATE,FEE,RENT_APPROVAL_YEAR,RENT_APPROVAL_MONTH
0,2025-03,kallang/whampoa,74,Whampoa Drive,3-room,improved,66.0,yes,1981,0.0,2025,03
1,2025-03,bukit timah,6,farrer road,4-room,improved,91.0,yes,1974,0.0,2025,03
2,2025-03,pasir ris,755,Pasir Ris Street 71,4-room,model a,105.0,yes,1996,0.0,2025,03
3,2025-03,punggol,299,punggol central,4-room,premium apartment,94.0,yes,2003,0.0,2025,03
4,2025-03,tampines,869a,tampines avenue 8,4-room,model a,92.0,yes,2015,0.0,2025,03
...,...,...,...,...,...,...,...,...,...,...,...,...
49995,2026-07,bukit batok,138,bukit batok west avenue 6,3-room,new generation,68.0,yes,1984,0.0,2026,07
49996,2026-07,queenstown,88,dawson road,4-room,premium apartment,83.0,yes,2016,0.0,2026,07
49997,2026-07,toa payoh,235,lorong 8 toa payoh,3-room,improved,76.0,yes,1976,0.0,2026,07
49998,2026-07,bukit panjang,613,Bukit Panjang Ring Road,4-room,model a,90.0,yes,2002,0.0,2026,07


In [7]:
print('Train Dates')
print(train['RENT_APPROVAL_YEAR'].sort_values().drop_duplicates().tolist())
print(train['RENT_APPROVAL_MONTH'].sort_values().drop_duplicates().tolist())
print(f'Train date Min {train["RENT_APPROVAL_DATE"].min()}')
print(f'Train date Max {train["RENT_APPROVAL_DATE"].max()}')
print('========================================')
print('Test Dates')
print(test['RENT_APPROVAL_YEAR'].sort_values().drop_duplicates().tolist())
print(test['RENT_APPROVAL_MONTH'].sort_values().drop_duplicates().tolist())
print(f'Test date Min {test["RENT_APPROVAL_DATE"].min()}')
print(f'Test date Max {test["RENT_APPROVAL_DATE"].max()}')

Train Dates
['2021', '2022', '2023', '2024', '2025']
['01', '02', '03', '04', '05', '06', '07', '08', '09', '10', '11', '12']
Train date Min 2021-01
Train date Max 2025-03
Test Dates
['2025', '2026']
['01', '02', '03', '04', '05', '06', '07', '08', '09', '10', '11', '12']
Test date Min 2025-03
Test date Max 2026-07


# Location
- [EDA Summary](#eda-summary)
- [Base Analysis](#base-analysis)
- [Dates](#check-dates)
- [Location](#location)
- [Flat Characteristics](#flat-characteristics)
- [Lease Characteristics](#lease-characteristics)
- [Clean Function](#clean-function)
- [Join Composite Keys](#join-composite-keys)

The main keys distinguishing the HDB blocks are the following:
- `Town`, 52 unique values
- `Block`, 5380 Unique values
- `Street`, 1149 unique values

We do basic cleaning of these values by making them lower case. We verified the cleanliness by eyeball, sorting the lower cased values and ensuring no duplicates by hand.

In [8]:
# Basic Cleaning
train['TOWN'] = train['TOWN'].apply(lambda x: x.lower())
train['BLOCK'] = train['BLOCK'].apply(lambda x: x.lower())
train['STREET'] = train['STREET'].apply(lambda x: x.lower())

test['TOWN'] = test['TOWN'].apply(lambda x: x.lower())
test['BLOCK'] = test['BLOCK'].apply(lambda x: x.lower())
test['STREET'] = test['STREET'].apply(lambda x: x.lower())

# Check unique
trt = train[['TOWN', 'BLOCK', 'STREET',]]
trt['in_train'] = 1
tst = test[['TOWN', 'BLOCK', 'STREET',]]
tst['in_test'] = 1

# Town
town_unique = pd.concat([trt[['TOWN']].drop_duplicates(),tst[['TOWN']].drop_duplicates()])
town_unique = pd.merge(town_unique, trt[['TOWN','in_train']].drop_duplicates(), how ='left', on = 'TOWN')
town_unique = pd.merge(town_unique, tst[['TOWN','in_test']].drop_duplicates(), how ='left', on = 'TOWN')
town_unique['in_train'] = town_unique['in_train'].fillna(0)
town_unique['in_test'] = town_unique['in_test'].fillna(0)
print('============ TOWN ============')
display(town_unique)

# Block
block_unique = pd.concat([trt[['BLOCK']].drop_duplicates(),tst[['BLOCK']].drop_duplicates()])
block_unique = pd.merge(block_unique, trt[['BLOCK','in_train']].drop_duplicates(), how ='left', on = 'BLOCK')
block_unique = pd.merge(block_unique, tst[['BLOCK','in_test']].drop_duplicates(), how ='left', on = 'BLOCK')
block_unique['in_train'] = block_unique['in_train'].fillna(0)
block_unique['in_test'] = block_unique['in_test'].fillna(0)
print('============ BLOCK ============')
display(block_unique)

# Street
street_unique = pd.concat([trt[['STREET']].drop_duplicates(),tst[['STREET']].drop_duplicates()])
street_unique = pd.merge(street_unique, trt[['STREET','in_train']].drop_duplicates(), how ='left', on = 'STREET')
street_unique = pd.merge(street_unique, tst[['STREET','in_test']].drop_duplicates(), how ='left', on = 'STREET')
street_unique['in_train'] = street_unique['in_train'].fillna(0)
street_unique['in_test'] = street_unique['in_test'].fillna(0)
print('============ STREET ============')
display(street_unique)

============ TOWN ============


/var/folders/j9/6y3df2850854pj09qr79r10w0000gn/T/ipykernel_27227/3708513278.py:12: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  trt['in_train'] = 1
/var/folders/j9/6y3df2850854pj09qr79r10w0000gn/T/ipykernel_27227/3708513278.py:14: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  tst['in_test'] = 1


,TOWN,in_train,in_test
0,punggol,1,1
1,sengkang,1,1
2,queenstown,1,1
3,sembawang,1,1
4,bukit merah,1,1
5,hougang,1,1
6,bedok,1,1
7,yishun,1,1
8,bukit panjang,1,1
9,toa payoh,1,1


============ BLOCK ============


,BLOCK,in_train,in_test
0,604a,1.0,1.0
1,290c,1.0,1.0
2,5,1.0,1.0
3,436b,1.0,1.0
4,467b,1.0,1.0
...,...,...,...
5375,342c,1.0,1.0
5376,376b,1.0,1.0
5377,513b,1.0,1.0
5378,262c,1.0,1.0


============ STREET ============


,STREET,in_train,in_test
0,punggol road,1,1.0
1,compassvale crescent,1,1.0
2,ghim moh road,1,1.0
3,fernvale road,1,1.0
4,admiralty drive,1,1.0
...,...,...,...
1144,kampong arang road,1,1.0
1145,aljunied avenue 2,1,1.0
1146,ang mo kio street 23,1,1.0
1147,seng poh road,1,1.0


In [9]:
# Eyeball to check for any weird location (Nothing weird)
print('============ TRAIN SET ============')
print(f"Unique Towns ({train['TOWN'].sort_values().unique().shape[0]}):{train['TOWN'].sort_values().unique().tolist()}")
print(f"Unique Street ({train['STREET'].sort_values().unique().shape[0]}):{train['STREET'].sort_values().unique().tolist()}")
print(f"Unique Blocks ({train['BLOCK'].sort_values().unique().shape[0]}):{train['BLOCK'].sort_values().unique().tolist()}")

print('============ TEST SET ============')
print(f"Unique Towns ({test['TOWN'].sort_values().unique().shape[0]}):{test['TOWN'].sort_values().unique().tolist()}")
print(f"Unique Street ({test['STREET'].sort_values().unique().shape[0]}):{test['STREET'].sort_values().unique().tolist()}")
print(f"Unique Blocks ({test['BLOCK'].sort_values().unique().shape[0]}):{test['BLOCK'].sort_values().unique().tolist()}")

============ TRAIN SET ============
Unique Towns (26):['ang mo kio', 'bedok', 'bishan', 'bukit batok', 'bukit merah', 'bukit panjang', 'bukit timah', 'central', 'choa chu kang', 'clementi', 'geylang', 'hougang', 'jurong east', 'jurong west', 'kallang/whampoa', 'marine parade', 'pasir ris', 'punggol', 'queenstown', 'sembawang', 'sengkang', 'serangoon', 'tampines', 'toa payoh', 'woodlands', 'yishun']
Unique Street (576):['admiralty drive', 'admiralty link', 'ah hood road', 'aljunied avenue 2', 'aljunied crescent', 'aljunied road', 'alkaff crescent', 'anchorvale crescent', 'anchorvale drive', 'anchorvale lane', 'anchorvale link', 'anchorvale road', 'anchorvale street', 'ang mo kio avenue 1', 'ang mo kio avenue 10', 'ang mo kio avenue 2', 'ang mo kio avenue 3', 'ang mo kio avenue 4', 'ang mo kio avenue 5', 'ang mo kio avenue 6', 'ang mo kio avenue 8', 'ang mo kio avenue 9', 'ang mo kio street 11', 'ang mo kio street 21', 'ang mo kio street 23', 'ang mo kio street 31', 'ang mo kio street 32

Here, we check if there are any rental blocks that are not in both the training dataset and test dataset. We find that there are some flats that don't exist in both sets.

As we intend to use geographical clustering to group flats within similar geographical regions in Singapore as a feature for our regression model, these missing values will affect our clustering result later. We will use the nearest cluster by distance to bridge the gap for these flats.

In [10]:
# Town exists for all regions.
display(town_unique[(town_unique['in_test'] == 1) & (town_unique['in_train'] == 0)])
display(town_unique[(town_unique['in_test'] == 0) & (town_unique['in_train'] == 1)])

,TOWN,in_train,in_test


,TOWN,in_train,in_test


In [11]:
# There are some missing Blocks
display(block_unique[(block_unique['in_test'] == 1) & (block_unique['in_train'] == 0)])
display(block_unique[(block_unique['in_test'] == 0) & (block_unique['in_train'] == 1)])

,BLOCK,in_train,in_test
4148,818c,0.0,1.0
4162,994a,0.0,1.0
4980,478c,0.0,1.0
5102,464b,0.0,1.0
5209,886d,0.0,1.0
5283,421a,0.0,1.0
5331,620a,0.0,1.0
5335,419b,0.0,1.0
5353,154a,0.0,1.0
5371,606d,0.0,1.0


,BLOCK,in_train,in_test
923,891b,1.0,0.0
1153,893d,1.0,0.0
1279,635c,1.0,0.0
1352,265a,1.0,0.0
1492,486a,1.0,0.0
...,...,...,...
2685,461c,1.0,0.0
2697,801c,1.0,0.0
2701,676d,1.0,0.0
2711,803c,1.0,0.0


In [12]:
# There are some missing Streets
display(street_unique[(street_unique['in_test'] == 1) & (street_unique['in_train'] == 0)])
display(street_unique[(street_unique['in_test'] == 0) & (street_unique['in_train'] == 1)])

,STREET,in_train,in_test


,STREET,in_train,in_test
437,macpherson lane,1,0.0
534,jalan ma'mor,1,0.0
573,margaret drive,1,0.0


# Flat Characteristics

- [EDA Summary](#eda-summary)
- [Base Analysis](#base-analysis)
- [Dates](#check-dates)
- [Location](#location)
- [Flat Characteristics](#flat-characteristics)
- [Lease Characteristics](#lease-characteristics)
- [Clean Function](#clean-function)
- [Join Composite Keys](#join-composite-keys)

We clean the unit level rents in this portion of the EDA. We have the following findings:
1. There are some duplicates for the `FLAT_TYPE`, caused by string formatting.
2. The `FLAT_MODEL` column looks alright, but we do not know much about the set of possible flat models. Instead, we ensure that there are not string formatting problems and simply used the HDB defined flat models as the ground truth.
3. `FLOOR_AREA_SQM` seems fine, with no weird outliers or potential missing value encoding
4. `Furnished` is a useless column as all the values are 'yes'


In [13]:
# Some duplicate flat types here
trt = train[[ 'FLAT_TYPE','FLAT_MODEL', 'FLOOR_AREA_SQM', 'FURNISHED',]]
tst = test[[ 'FLAT_TYPE','FLAT_MODEL', 'FLOOR_AREA_SQM', 'FURNISHED',]]
trt.groupby('FLAT_TYPE').count()

,FLAT_MODEL,FLOOR_AREA_SQM,FURNISHED
FLAT_TYPE,,,
1 room,8,8,8
1-room,34,34,34
2 room,532,532,532
2-room,2566,2566,2566
3 room,12092,12092,12092
3-room,36932,36932,36932
4 room,13391,13391,13391
4-room,40688,40688,40688
5 room,9308,9308,9308


In [14]:
trt['flat_type_processed'] = trt['FLAT_TYPE'].apply(lambda x: x.replace('-', ' '))
trt.groupby('flat_type_processed').count()

/var/folders/j9/6y3df2850854pj09qr79r10w0000gn/T/ipykernel_27227/65275034.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  trt['flat_type_processed'] = trt['FLAT_TYPE'].apply(lambda x: x.replace('-', ' '))


,FLAT_TYPE,FLAT_MODEL,FLOOR_AREA_SQM,FURNISHED
flat_type_processed,,,,
1 room,42,42,42,42
2 room,3098,3098,3098,3098
3 room,49024,49024,49024,49024
4 room,54079,54079,54079,54079
5 room,35523,35523,35523,35523
executive,8234,8234,8234,8234


In [15]:
# Some incorrect flat models here, need to clean
print('============ TRAIN SET ============')
display(trt.groupby('FLAT_MODEL').count().sort_values('FLAT_MODEL'))
print('============ TEST SET ============')
display(tst.groupby('FLAT_MODEL').count().sort_values('FLAT_MODEL'))

============ TRAIN SET ============


,FLAT_TYPE,FLOOR_AREA_SQM,FURNISHED,flat_type_processed
FLAT_MODEL,,,,
2 room,224,224,224,224
3gen,12,12,12,12
adjoined flat,218,218,218,218
apartment,4724,4724,4724,4724
dbss,1090,1090,1090,1090
improved,44008,44008,44008,44008
improved maisonette,1,1,1,1
maisonette,2346,2346,2346,2346
model a,42745,42745,42745,42745


============ TEST SET ============


,FLAT_TYPE,FLOOR_AREA_SQM,FURNISHED
FLAT_MODEL,,,
2 room,235,235,235
3gen,8,8,8
adjoined flat,93,93,93
apartment,1488,1488,1488
dbss,399,399,399
improved,14429,14429,14429
maisonette,694,694,694
model a,14923,14923,14923
model a maisonette,45,45,45


In [16]:
# According to HDB, the following are the available models in existence for flats
hdb_flat_models = [
    "Standard",
    "Improved",
    "New Generation",
    "Simplified",
    "Model A",
    "Model A2",
    "Apartment",
    "Premium Apartment",
    "Premium Apartment Loft",
    "Maisonette",
    "Model A-Maisonette",
    "Premium Maisonette",
    "Improved-Maisonette",
    "Adjoined Flat",
    "Multi-Generation",
    "DBSS",
    "Terrace",
    "Type S1",
    "Type S2",
    "2-room"
]

In [17]:
# We can just drop this column
trt.groupby('FURNISHED').count().sort_values('FURNISHED')

,FLAT_TYPE,FLAT_MODEL,FLOOR_AREA_SQM,flat_type_processed
FURNISHED,,,,
yes,150000,150000,150000,150000


In [18]:
# No null values, no weird float values
display(trt[['FLOOR_AREA_SQM']].describe())
trt[trt['FLOOR_AREA_SQM'].notnull()]

,FLOOR_AREA_SQM
count,150000.000000
mean,93.338847
std,24.271768
min,31.000000
25%,68.000000
50%,92.000000
75%,110.000000
max,215.000000


,FLAT_TYPE,FLAT_MODEL,FLOOR_AREA_SQM,FURNISHED,flat_type_processed
0,4 room,model a,93.0,yes,4 room
1,5 room,improved,110.0,yes,5 room
2,3 room,improved,65.0,yes,3 room
3,4 room,model a,96.0,yes,4 room
4,4 room,premium apartment,102.0,yes,4 room
...,...,...,...,...,...
149995,3-room,new generation,67.0,yes,3 room
149996,5-room,improved,107.0,yes,5 room
149997,3-room,improved,67.0,yes,3 room
149998,4-room,premium apartment,94.0,yes,4 room


# Lease Characteristics

- [EDA Summary](#eda-summary)
- [Base Analysis](#base-analysis)
- [Dates](#check-dates)
- [Location](#location)
- [Flat Characteristics](#flat-characteristics)
- [Lease Characteristics](#lease-characteristics)
- [Clean Function](#clean-function)
- [Join Composite Keys](#join-composite-keys)

Here we check the lease date, the fee and monthly rent. The findings are:
1. Lease is the year the block was completed. It is some integer. There is one row where the year of rent is more than the year of lease, which makes no sense, so we drop it.
2. The Fee is useless as it is all 0, so we are dropping it.
3. The monthly rent looks fine.

In [19]:
# Lease Commence Date is a number here, we want it to be a string type
# Fee is useless, can drop
# Rent looks ok
train[['LEASE_COMMENCE_DATE','FEE', 'MONTHLY_RENT',]].describe()

,LEASE_COMMENCE_DATE,FEE,MONTHLY_RENT
count,150000.000000,150000.0,150000.000000
mean,1990.907473,0.0,2719.036893
std,12.641540,0.0,749.731096
min,1966.000000,0.0,300.000000
25%,1981.000000,0.0,2100.000000
50%,1988.000000,0.0,2700.000000
75%,2001.000000,0.0,3200.000000
max,2022.000000,0.0,7600.000000


In [20]:
# Check by eyeball, all looks ok
print("Unique Lease Years")
train['LEASE_COMMENCE_DATE'] = train['LEASE_COMMENCE_DATE'].apply(lambda x: int(x))
test['LEASE_COMMENCE_DATE'] = test['LEASE_COMMENCE_DATE'].apply(lambda x: int(x))
print('============ TRAIN SET ============')
print(sorted(train['LEASE_COMMENCE_DATE'].unique().tolist()))
print('============ TEST SET ============')
print(sorted(test['LEASE_COMMENCE_DATE'].unique().tolist()))

Unique Lease Years
============ TRAIN SET ============
[1966, 1967, 1968, 1969, 1970, 1971, 1972, 1973, 1974, 1975, 1976, 1977, 1978, 1979, 1980, 1981, 1982, 1983, 1984, 1985, 1986, 1987, 1988, 1989, 1990, 1991, 1992, 1993, 1994, 1995, 1996, 1997, 1998, 1999, 2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022]
============ TEST SET ============
[1966, 1967, 1968, 1969, 1970, 1971, 1972, 1973, 1974, 1975, 1976, 1977, 1978, 1979, 1980, 1981, 1982, 1983, 1984, 1985, 1986, 1987, 1988, 1989, 1990, 1991, 1992, 1993, 1994, 1995, 1996, 1997, 1998, 1999, 2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022]


In [21]:
# Train set has row with wrong data, so we remove it.
print('============ TRAIN SET ============')
print("Set of train/test rows where the lease year is after the rent approval year")
display(train[train['LEASE_COMMENCE_DATE'] > train['RENT_APPROVAL_YEAR'].apply(lambda x: int(x))])
print('============ TEST SET ============')
display(test[test['LEASE_COMMENCE_DATE'] > test['RENT_APPROVAL_YEAR'].apply(lambda x: int(x))])

============ TRAIN SET ============
Set of train/test rows where the lease year is after the rent approval year


,RENT_APPROVAL_DATE,TOWN,BLOCK,STREET,FLAT_TYPE,FLAT_MODEL,FLOOR_AREA_SQM,FURNISHED,LEASE_COMMENCE_DATE,FEE,MONTHLY_RENT,RENT_APPROVAL_YEAR,RENT_APPROVAL_MONTH
31739,2021-11,bedok,153b,bedok south road,5 room,improved,113.0,yes,2022,0.0,3200,2021,11


============ TEST SET ============


,RENT_APPROVAL_DATE,TOWN,BLOCK,STREET,FLAT_TYPE,FLAT_MODEL,FLOOR_AREA_SQM,FURNISHED,LEASE_COMMENCE_DATE,FEE,RENT_APPROVAL_YEAR,RENT_APPROVAL_MONTH


# Clean Function
- [EDA Summary](#eda-summary)
- [Base Analysis](#base-analysis)
- [Dates](#check-dates)
- [Location](#location)
- [Flat Characteristics](#flat-characteristics)
- [Lease Characteristics](#lease-characteristics)
- [Clean Function](#clean-function)
- [Join Composite Keys](#join-composite-keys)


Steps:
1. Drop `FEE` and `FURNISHED`
2. Process Dates `RENT_APPROVAL_DATE` and `LEASE_COMMENCE_DATE`
3. Remove training row where the `RENT_APPROVAL_YEAR` > `LEASE_COMMENCE_DATE`
4. String processing for `TOWN`, `BLOCK` and `STREET`
5. Clean `FLAT_TYPE`  

In [22]:
import pandas as pd
import numpy as np
import scipy as sp
import os, sys
from pathlib import Path
_here = Path(os.environ.get('RENT_PROJECT_ROOT') or Path.cwd()).resolve()
sys.path.insert(0, str(next(p for p in [_here, *_here.parents] if (p / 'data' / 'config.py').exists())))      # final_report: data/config.py holds every path
from data import config as cfg
ROOT = cfg.ROOT

In [23]:
def clean_data(df):
    res_df = df.copy(deep = True)

    # 1. Remove FEE and FURNISHED
    res_df = res_df.drop(['FEE','FURNISHED'], axis = 1)

    # 2. Process Rent Approval Date
    res_df['RENT_APPROVAL_YEAR'] = res_df['RENT_APPROVAL_DATE'].apply(lambda x: x.split('-')[0]).astype('int')    
    res_df['RENT_APPROVAL_MONTH'] = res_df['RENT_APPROVAL_DATE'].apply(lambda x: x.split('-')[1])
    res_df = res_df.drop(['RENT_APPROVAL_DATE'], axis = 1)

    # Process Lease Date
    res_df['LEASE_COMMENCE_DATE'] = res_df['LEASE_COMMENCE_DATE'].astype('int')

    # 3. Remove RENT_APPROVAL_YEAR > LEASE_COMMENCE_DATE
    res_df = res_df[res_df['RENT_APPROVAL_YEAR'] >= res_df['LEASE_COMMENCE_DATE']]
    res_df['RENT_APPROVAL_YEAR'] = res_df['RENT_APPROVAL_YEAR'].astype('str')
    res_df['LEASE_COMMENCE_DATE'] = res_df['LEASE_COMMENCE_DATE'].astype('str')

    # 4. Town Block and Street Processing
    res_df['TOWN'] = res_df['TOWN'].apply(lambda x: x.lower())
    res_df['BLOCK'] = res_df['BLOCK'].apply(lambda x: x.lower())
    res_df['STREET'] = res_df['STREET'].apply(lambda x: x.lower())

    # 5. Clean Flat Type
    res_df['FLAT_TYPE'] = res_df['FLAT_TYPE'].apply(lambda x: x.replace('-', ' ').lower())

    col_select = ['RENT_APPROVAL_YEAR',
       'RENT_APPROVAL_MONTH', 'LEASE_COMMENCE_DATE', 'TOWN', 'BLOCK', 'STREET', 'FLAT_TYPE', 'FLAT_MODEL', 'FLOOR_AREA_SQM', 
       'MONTHLY_RENT', ]

    if 'MONTHLY_RENT' not in res_df.columns:
        col_select = col_select[:-1]

    res_df = res_df[col_select].reset_index(drop = True)

    return res_df

In [24]:
train = pd.read_csv(f'{cfg.ORIGINAL_DIR}/train.csv')
test = pd.read_csv(f'{cfg.ORIGINAL_DIR}/test.csv')
df_train = clean_data(train)
df_test = clean_data(test)
df_train

,RENT_APPROVAL_YEAR,RENT_APPROVAL_MONTH,LEASE_COMMENCE_DATE,TOWN,BLOCK,STREET,FLAT_TYPE,FLAT_MODEL,FLOOR_AREA_SQM,MONTHLY_RENT
0,2021,01,2012,punggol,604a,punggol road,4 room,model a,93.0,1800
1,2021,01,2001,sengkang,290c,compassvale crescent,5 room,improved,110.0,2000
2,2021,01,1976,queenstown,5,ghim moh road,3 room,improved,65.0,2000
3,2021,01,2010,sengkang,436b,fernvale road,4 room,model a,96.0,1400
4,2021,01,2001,sembawang,467b,admiralty drive,4 room,premium apartment,102.0,1800
...,...,...,...,...,...,...,...,...,...,...
149994,2025,03,1985,bukit batok,220,bukit batok east avenue 3,3 room,new generation,67.0,2500
149995,2025,03,2020,queenstown,79,dawson road,5 room,improved,107.0,5300
149996,2025,03,1971,toa payoh,17,lorong 7 toa payoh,3 room,improved,67.0,2650
149997,2025,03,2001,sembawang,360a,admiralty drive,4 room,premium apartment,94.0,3300


In [10]:
df_test

,RENT_APPROVAL_YEAR,RENT_APPROVAL_MONTH,LEASE_COMMENCE_DATE,TOWN,BLOCK,STREET,FLAT_TYPE,FLAT_MODEL,FLOOR_AREA_SQM
0,2025,03,1981,kallang/whampoa,74,whampoa drive,3 room,improved,66.0
1,2025,03,1974,bukit timah,6,farrer road,4 room,improved,91.0
2,2025,03,1996,pasir ris,755,pasir ris street 71,4 room,model a,105.0
3,2025,03,2003,punggol,299,punggol central,4 room,premium apartment,94.0
4,2025,03,2015,tampines,869a,tampines avenue 8,4 room,model a,92.0
...,...,...,...,...,...,...,...,...,...
49995,2026,07,1984,bukit batok,138,bukit batok west avenue 6,3 room,new generation,68.0
49996,2026,07,2016,queenstown,88,dawson road,4 room,premium apartment,83.0
49997,2026,07,1976,toa payoh,235,lorong 8 toa payoh,3 room,improved,76.0
49998,2026,07,2002,bukit panjang,613,bukit panjang ring road,4 room,model a,90.0


# Join Composite Keys
- [EDA Summary](#eda-summary)
- [Base Analysis](#base-analysis)
- [Dates](#check-dates)
- [Location](#location)
- [Flat Characteristics](#flat-characteristics)
- [Lease Characteristics](#lease-characteristics)
- [Clean Function](#clean-function)
- [Join Composite Keys](#join-composite-keys)

Here we join the HDB dataset to get some extra composite keys to join our features later.

In [50]:
hdb = pd.read_csv(f'{cfg.AUXILIARY_DIR}/sg-hdb-block.csv')
hdb

,TOWN,BLOCK,STREET,POSTAL_CODE,LATITUDE,LONGITUDE,MAX_FLOOR,YEAR_COMPLETED,SUBZONE,PLANNING_AREA,REGION
0,serangoon,106,serangoon north avenue 1,550106,1.370245,103.870823,12,1983,serangoon garden,serangoon,north-east region
1,bukit merah,34,telok blangah way,090034,1.273518,103.822481,14,1975,telok blangah rise,bukit merah,central region
2,punggol,226b,sumang lane,822226,1.402685,103.893609,17,2016,matilda,punggol,north-east region
3,marine parade,74,marine drive,440074,1.303705,103.907214,14,1975,marine parade,marine parade,central region
4,choa chu kang,284,choa chu kang avenue 3,NIL,1.379714,103.742138,16,1990,keat hong,choa chu kang,west region
...,...,...,...,...,...,...,...,...,...,...,...
9459,toa payoh,155,lorong 1 toa payoh,310155,1.332796,103.845589,30,1998,toa payoh central,toa payoh,central region
9460,sembawang,307,canberra road,750307,1.445923,103.821657,16,1997,sembawang central,sembawang,north region
9461,woodlands,681a,woodlands drive 62,731681,1.439534,103.803797,14,1998,woodlands east,woodlands,north region
9462,choa chu kang,673a,choa chu kang crescent,681673,1.401014,103.747524,16,1998,yew tee,choa chu kang,west region


In [51]:
# Check the HDB dataset is in the train data
df2 = df_train[['TOWN','BLOCK','STREET']]
hdb2 = hdb[['TOWN','BLOCK','STREET']]
hdb2['in_hdb'] = 1
dd = pd.merge(df2, hdb2, how = 'left', on = ['TOWN','BLOCK','STREET'])
dd.count()

/var/folders/j9/6y3df2850854pj09qr79r10w0000gn/T/ipykernel_5252/310266714.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  hdb2['in_hdb'] = 1


TOWN      149999
BLOCK     149999
STREET    149999
in_hdb    149999
dtype: int64

In [58]:
# Check the HDB dataset is in the test data
df2 = df_test[['TOWN','BLOCK','STREET']]
hdb2 = hdb[['TOWN','BLOCK','STREET']]
hdb2['in_hdb'] = 1
dd = pd.merge(df2, hdb2, how = 'left', on = ['TOWN','BLOCK','STREET'])
dd.count()

/var/folders/j9/6y3df2850854pj09qr79r10w0000gn/T/ipykernel_5252/1670155232.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  hdb2['in_hdb'] = 1


TOWN      50000
BLOCK     50000
STREET    50000
in_hdb    50000
dtype: int64

In [53]:
train_df = pd.merge(df_train, hdb, how = 'left', on = ['TOWN','BLOCK','STREET'])
train_df.to_csv(f'{cfg.DF_KEYS_DIR}/train-house.csv', index = False)
train_df

,RENT_APPROVAL_YEAR,RENT_APPROVAL_MONTH,LEASE_COMMENCE_DATE,TOWN,BLOCK,STREET,FLAT_TYPE,FLAT_MODEL,FLOOR_AREA_SQM,MONTHLY_RENT,POSTAL_CODE,LATITUDE,LONGITUDE,MAX_FLOOR,YEAR_COMPLETED,SUBZONE,PLANNING_AREA,REGION
0,2021,01,2012,punggol,604a,punggol road,4 room,model a,93.0,1800,821604,1.404310,103.907413,16,2010,waterway east,punggol,north-east region
1,2021,01,2001,sengkang,290c,compassvale crescent,5 room,improved,110.0,2000,543290,1.397018,103.897535,15,2000,sengkang town centre,sengkang,north-east region
2,2021,01,1976,queenstown,5,ghim moh road,3 room,improved,65.0,2000,270005,1.312693,103.788309,15,1975,ghim moh,queenstown,central region
3,2021,01,2010,sengkang,436b,fernvale road,4 room,model a,96.0,1400,792436,1.392878,103.876976,21,2008,fernvale,sengkang,north-east region
4,2021,01,2001,sembawang,467b,admiralty drive,4 room,premium apartment,102.0,1800,752467,1.451141,103.813630,16,1999,sembawang north,sembawang,north region
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
149994,2025,03,1985,bukit batok,220,bukit batok east avenue 3,3 room,new generation,67.0,2500,650220,1.347456,103.754710,4,1984,bukit batok central,bukit batok,west region
149995,2025,03,2020,queenstown,79,dawson road,5 room,improved,107.0,5300,141079,1.294591,103.810327,39,2019,margaret drive,queenstown,central region
149996,2025,03,1971,toa payoh,17,lorong 7 toa payoh,3 room,improved,67.0,2650,310017,1.336427,103.856608,20,1970,pei chun,toa payoh,central region
149997,2025,03,2001,sembawang,360a,admiralty drive,4 room,premium apartment,94.0,3300,751360,1.449203,103.815848,16,1999,sembawang central,sembawang,north region


In [54]:
train_df[train_df['SUBZONE'].isna()]

,RENT_APPROVAL_YEAR,RENT_APPROVAL_MONTH,LEASE_COMMENCE_DATE,TOWN,BLOCK,STREET,FLAT_TYPE,FLAT_MODEL,FLOOR_AREA_SQM,MONTHLY_RENT,POSTAL_CODE,LATITUDE,LONGITUDE,MAX_FLOOR,YEAR_COMPLETED,SUBZONE,PLANNING_AREA,REGION


In [56]:
test_df = pd.merge(df_test, hdb, how = 'left', on = ['TOWN','BLOCK','STREET'])
test_df.to_csv(f'{cfg.DF_KEYS_DIR}/test-house.csv', index = False)
test_df

,RENT_APPROVAL_YEAR,RENT_APPROVAL_MONTH,LEASE_COMMENCE_DATE,TOWN,BLOCK,STREET,FLAT_TYPE,FLAT_MODEL,FLOOR_AREA_SQM,POSTAL_CODE,LATITUDE,LONGITUDE,MAX_FLOOR,YEAR_COMPLETED,SUBZONE,PLANNING_AREA,REGION
0,2025,03,1981,kallang/whampoa,74,whampoa drive,3 room,improved,66.0,320074,1.324716,103.857753,16,1973,balestier,novena,central region
1,2025,03,1974,bukit timah,6,farrer road,4 room,improved,91.0,260006,1.317320,103.808057,10,1973,tyersall,tanglin,central region
2,2025,03,1996,pasir ris,755,pasir ris street 71,4 room,model a,105.0,510755,1.377471,103.933704,16,1994,pasir ris west,pasir ris,east region
3,2025,03,2003,punggol,299,punggol central,4 room,premium apartment,94.0,820299,1.402706,103.905308,16,2002,matilda,punggol,north-east region
4,2025,03,2015,tampines,869a,tampines avenue 8,4 room,model a,92.0,521869,1.355018,103.932317,16,2014,tampines west,tampines,east region
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
49995,2026,07,1984,bukit batok,138,bukit batok west avenue 6,3 room,new generation,68.0,650138,1.350447,103.744472,12,1983,bukit batok west,bukit batok,west region
49996,2026,07,2016,queenstown,88,dawson road,4 room,premium apartment,83.0,142088,1.295794,103.809265,47,2015,margaret drive,queenstown,central region
49997,2026,07,1976,toa payoh,235,lorong 8 toa payoh,3 room,improved,76.0,310235,1.338950,103.857139,12,1975,pei chun,toa payoh,central region
49998,2026,07,2002,bukit panjang,613,bukit panjang ring road,4 room,model a,90.0,670613,1.384367,103.762046,21,2000,senja,bukit panjang,west region


In [57]:
test_df[test_df['SUBZONE'].isna()]

,RENT_APPROVAL_YEAR,RENT_APPROVAL_MONTH,LEASE_COMMENCE_DATE,TOWN,BLOCK,STREET,FLAT_TYPE,FLAT_MODEL,FLOOR_AREA_SQM,POSTAL_CODE,LATITUDE,LONGITUDE,MAX_FLOOR,YEAR_COMPLETED,SUBZONE,PLANNING_AREA,REGION
